# FinSight – Model Training & Evaluation

Trains all models and displays the evaluation results.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

print('Libraries ready')

In [ ]:
from ml.feature_engineering import load_and_engineer, time_based_split

print('Loading and engineering features...')
X, y, feature_cols = load_and_engineer(Path('../data/raw'))
X_train, X_val, y_train, y_val = time_based_split(X, y)

print(f'Features: {len(feature_cols)}')
print(f'Train: {len(X_train):,} ({y_train.mean():.2%} fraud)')
print(f'Val:   {len(X_val):,} ({y_val.mean():.2%} fraud)')

In [ ]:
from ml.training import train_all_models, select_best_model

print('Training all models (this may take several minutes)...')
trained_models, all_metrics = train_all_models(X_train, y_train, X_val, y_val)

# Display comparison
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.sort_values('pr_auc', ascending=False)
print('\nModel Comparison:')
display(metrics_df.style.highlight_max(color='lightgreen').format('{:.4f}'))

In [ ]:
from ml.evaluation import plot_model_comparison
plot_model_comparison(all_metrics, Path('../reports/evaluation'))
plt.show()

In [ ]:
from ml.training import optimise_best_model, calibrate_model
from ml.evaluation import generate_evaluation_report

best_name = select_best_model(all_metrics)
print(f'Best model: {best_name}')

# Optimise with Optuna
print('Running Optuna optimisation (30 trials)...')
optimised = optimise_best_model(best_name, X_train, y_train, X_val, y_val, n_trials=30)

if optimised is not None:
    trained_models[best_name] = optimised

# Calibrate
best_model = calibrate_model(trained_models[best_name], X_val, y_val)
y_prob = best_model.predict_proba(X_val)[:, 1]

from ml.training import _compute_metrics
final_metrics = _compute_metrics(y_val.values, y_prob)
print('\nFinal metrics:')
for k, v in final_metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
from ml.evaluation import generate_evaluation_report

report = generate_evaluation_report(
    y_true=y_val.values,
    y_prob=y_prob,
    model=best_model,
    model_name=best_name,
    feature_cols=feature_cols,
    threshold=final_metrics['optimal_threshold'],
    output_dir=Path('../reports/evaluation'),
    all_metrics=all_metrics,
)

# Display plots
from PIL import Image
for plot_name in ['roc_curve', 'pr_curve', 'confusion_matrix', 'score_distribution']:
    img_path = Path(f'../reports/evaluation/{plot_name}.png')
    if img_path.exists():
        img = Image.open(img_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(plot_name.replace('_', ' ').title())
        plt.show()

In [ ]:
from ml.explainability import generate_global_explanations

print('Computing SHAP explanations (sampling 500 rows)...')
shap_results = generate_global_explanations(
    model=best_model,
    X_val=X_val.sample(min(500, len(X_val)), random_state=42),
    output_dir=Path('../artifacts/plots/shap'),
)

print('\nTop 10 features by global SHAP importance:')
for feat, val in list(shap_results['global_importance'].items())[:10]:
    print(f'  {feat}: {val:.6f}')

In [ ]:
from ml.training import persist_model_artifacts

paths = persist_model_artifacts(
    model=best_model,
    scaler=None,
    feature_cols=feature_cols,
    metrics=final_metrics,
    optimal_threshold=final_metrics['optimal_threshold'],
    model_name=best_name,
    output_dir=Path('../models'),
)

print('Model artifacts saved:')
for k, v in paths.items():
    print(f'  {k}: {v}')